# Section 3 — Quantization: fp16 vs 4-bit NF4 trade-off

**Electro Pi AI Engineer Test — Task 3.1**

This notebook runs a small open-weight LLM twice — once at **fp16** (full precision)
and once at **4-bit NF4** (quantized with `bitsandbytes`) — and measures:

- **Memory footprint** — VRAM used by the weights, and peak VRAM during generation
- **Throughput** — tokens/second on a fixed set of 5 prompts
- **Quality** — the actual text output for the same 5 prompts, side by side

It ends with a filled-in trade-off table and a half-page write-up comparing
GPTQ / AWQ / GGUF / bitsandbytes for production.

---

### Before you run
1. **Runtime → Change runtime type → T4 GPU** (free tier is enough).
2. Run the cells top to bottom.
3. Default model is **`Qwen/Qwen2.5-1.5B-Instruct`** — it is *ungated*, so no
   Hugging Face login is needed. If you switch to a Llama model you must accept
   its license on the HF model page and add an HF token (see the note in the config cell).


## 1. Install dependencies

In [1]:
%pip install -q -U transformers accelerate bitsandbytes
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
done


## 2. Confirm the GPU is available

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

CUDA available: True
GPU: Tesla T4
Tesla T4, 15360 MiB


## 3. Configuration (model, prompts, generation settings)

In [3]:
import time, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- model -----------------------------------------------------------------
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # ungated, ~1.5B params, fits a free T4
# Alternatives (Llama needs license acceptance + HF token):
#   MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
#   from huggingface_hub import login; login("hf_...")

# --- generation (kept identical for both runs so the comparison is fair) ----
MAX_NEW_TOKENS = 128
GEN_KWARGS = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)  # greedy = deterministic

# --- 5 FIXED prompts (same for fp16 and 4-bit) -----------------------------
PROMPTS = [
    "Explain what a transformer neural network is, in two sentences.",
    "What is the capital of Australia, and name two famous landmarks there?",
    "Write a Python function that returns the nth Fibonacci number.",
    "A farmer has 17 sheep. All but 9 run away. How many sheep are left? Explain briefly.",
    "In one sentence, summarize why quantization is useful for deploying LLMs.",
]

# tokenizer is shared by both runs
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model: {MODEL_ID}")
print(f"{len(PROMPTS)} fixed prompts | max_new_tokens={MAX_NEW_TOKENS} | greedy decoding")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-1.5B-Instruct
5 fixed prompts | max_new_tokens=128 | greedy decoding


## 4. Helper functions

`run_config` loads the model in the requested precision, does an untimed warm-up
(the first CUDA call is always slow and would skew the numbers), then times
generation over the 5 prompts and records VRAM.

In [4]:
def load_model(quantized: bool):
    """Load the model in fp16, or in 4-bit NF4 with bitsandbytes."""
    if quantized:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",              # NF4 = normal-float 4-bit
            bnb_4bit_compute_dtype=torch.float16,   # matmuls run in fp16
            bnb_4bit_use_double_quant=True,         # extra compression of the scales
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, quantization_config=bnb_config, device_map="cuda",
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype=torch.float16, device_map="cuda",
        )
    model.eval()
    return model


def generate(model, prompt, max_new_tokens):
    """Run one prompt through the chat template and return (text, n_new_tokens, seconds)."""
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,          # returns input_ids + attention_mask (dict-like)
    ).to("cuda")
    input_len = inputs["input_ids"].shape[1]   # read length from the dict

    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,                            # unpack ids + attention mask
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    dt = time.time() - t0

    n_new = out.shape[1] - input_len
    text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
    return text.strip(), n_new, dt


def run_config(name, quantized):
    """Full pass: load -> measure weights VRAM -> warm-up -> timed run -> peak VRAM."""
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

    t0 = time.time()
    model = load_model(quantized)
    load_time = time.time() - t0
    weights_vram = torch.cuda.memory_allocated() / 1e9   # GB used by the weights

    # untimed warm-up so the first-call overhead doesn't pollute throughput
    _ = generate(model, PROMPTS[0], max_new_tokens=16)

    torch.cuda.reset_peak_memory_stats()
    outputs, total_tokens, total_time = [], 0, 0.0
    for p in PROMPTS:
        text, n_new, dt = generate(model, p, MAX_NEW_TOKENS)
        outputs.append(text)
        total_tokens += n_new
        total_time += dt

    peak_vram = torch.cuda.max_memory_allocated() / 1e9   # weights + activations/KV cache
    tok_per_sec = total_tokens / total_time

    result = {
        "config": name,
        "weights_VRAM_GB": round(weights_vram, 2),
        "peak_VRAM_GB": round(peak_vram, 2),
        "tokens_per_sec": round(tok_per_sec, 1),
        "total_new_tokens": total_tokens,
        "total_gen_time_s": round(total_time, 2),
        "load_time_s": round(load_time, 1),
        "outputs": outputs,
    }

    del model
    gc.collect(); torch.cuda.empty_cache()
    return result

print("helpers ready")

helpers ready


## 5. Pass 1 — full precision (fp16)

We load fp16, benchmark it, then free the GPU memory **before** loading the 4-bit
version so the two measurements don't interfere.

In [5]:
res_fp16 = run_config("fp16 (baseline)", quantized=False)
print("fp16 weights VRAM :", res_fp16["weights_VRAM_GB"], "GB")
print("fp16 peak VRAM    :", res_fp16["peak_VRAM_GB"], "GB")
print("fp16 throughput   :", res_fp16["tokens_per_sec"], "tok/s")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

fp16 weights VRAM : 3.09 GB
fp16 peak VRAM    : 3.11 GB
fp16 throughput   : 16.3 tok/s


## 6. Pass 2 — 4-bit NF4 (bitsandbytes)

In [6]:
res_4bit = run_config("4-bit NF4 (bitsandbytes)", quantized=True)
print("4-bit weights VRAM:", res_4bit["weights_VRAM_GB"], "GB")
print("4-bit peak VRAM   :", res_4bit["peak_VRAM_GB"], "GB")
print("4-bit throughput  :", res_4bit["tokens_per_sec"], "tok/s")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


4-bit weights VRAM: 1.16 GB
4-bit peak VRAM   : 1.22 GB
4-bit throughput  : 10.0 tok/s


## 7. Trade-off table — precision vs. size vs. speed

The numbers below are produced from *your* run, so paste this table straight
into your README / NOTES.md.

In [7]:
import pandas as pd

def summary_row(r):
    return {
        "Config": r["config"],
        "Weights VRAM (GB)": r["weights_VRAM_GB"],
        "Peak VRAM (GB)": r["peak_VRAM_GB"],
        "Throughput (tok/s)": r["tokens_per_sec"],
        "Load time (s)": r["load_time_s"],
    }

df = pd.DataFrame([summary_row(res_fp16), summary_row(res_4bit)])

# relative deltas vs fp16 baseline
mem_saving = 100 * (1 - res_4bit["weights_VRAM_GB"] / res_fp16["weights_VRAM_GB"])
speed_ratio = res_4bit["tokens_per_sec"] / res_fp16["tokens_per_sec"]

print(df.to_string(index=False))
print()
print(f"Memory saving from 4-bit : {mem_saving:.0f}% smaller weights")
print(f"Speed ratio (4-bit/fp16) : {speed_ratio:.2f}x")
df

                  Config  Weights VRAM (GB)  Peak VRAM (GB)  Throughput (tok/s)  Load time (s)
         fp16 (baseline)               3.09            3.11                16.3           39.4
4-bit NF4 (bitsandbytes)               1.16            1.22                10.0           21.7

Memory saving from 4-bit : 62% smaller weights
Speed ratio (4-bit/fp16) : 0.61x


,Config,Weights VRAM (GB),Peak VRAM (GB),Throughput (tok/s),Load time (s)
0,fp16 (baseline),3.09,3.11,16.3,39.4
1,4-bit NF4 (bitsandbytes),1.16,1.22,10.0,21.7


## 8. Qualitative quality — same 5 prompts, both versions

Read these side by side and note any differences (missed facts, worse reasoning,
broken code, repetition). NF4 on a small model is usually very close to fp16, but
spot-check the reasoning prompt (#4) and the code prompt (#3) — those are where
degradation shows up first.

In [8]:
for i, prompt in enumerate(PROMPTS):
    print("=" * 90)
    print(f"PROMPT {i+1}: {prompt}")
    print("-" * 90)
    print(f"[fp16]  {res_fp16['outputs'][i]}")
    print("-" * 90)
    print(f"[4-bit] {res_4bit['outputs'][i]}")
    print()

PROMPT 1: Explain what a transformer neural network is, in two sentences.
------------------------------------------------------------------------------------------
[fp16]  A Transformer neural network is a type of deep learning architecture that uses self-attention mechanisms to process sequential data, enabling it to capture long-range dependencies and perform tasks such as language translation and natural language processing with high accuracy.
------------------------------------------------------------------------------------------
[4-bit] A transformer neural network is an advanced type of deep learning model that uses self-attention mechanisms to process and generate sequences of data more efficiently than traditional recurrent neural networks. It has been successfully applied in various tasks such as natural language processing, computer vision, and speech recognition.

PROMPT 2: What is the capital of Australia, and name two famous landmarks there?
----------------------------

### Your quality notes (fill this in)

Write 3–5 lines here after reading the outputs above. Example structure:

> On prompts 1, 2 and 5 the two versions were essentially identical. On the
> Fibonacci prompt the 4-bit version [matched / dropped an edge case / …].
> On the sheep reasoning prompt both answered **8** correctly. Overall the 4-bit
> model kept ~[X]% of the perceived quality while using ~[Y]% less VRAM.


## 9. Write-up (half page) — GPTQ / AWQ vs bitsandbytes, and when GGUF

*This is a reasoned answer you can keep or edit in your own voice.*

**bitsandbytes (what this notebook used).** Its strength is convenience: quantization
happens *at load time* with no calibration data and no offline step, and the same
4-bit weights can be fine-tuned with QLoRA. That makes it my default for
**experiments, research, and any workflow that involves training/fine-tuning**.
The cost is inference speed — dequantization happens on the fly during every
forward pass, so raw throughput is lower than formats that ship pre-packed
weights with fused kernels. It's also tied to a GPU + the PyTorch/transformers stack.

**GPTQ / AWQ.** These are *post-training* quantization methods that use a small
calibration dataset to quantize once, offline, into weights optimized for fast
inference kernels (ExLlama, Marlin). I'd pick them for **GPU-served production**
where you quantize a model once and then serve millions of requests — the one-time
calibration cost is amortized and you get meaningfully higher tokens/sec and better
batching than bitsandbytes. Between the two, **AWQ** tends to preserve quality
slightly better at 4-bit because it protects the most salient (activation-aware)
weight channels, so I lean AWQ when quality is sensitive; GPTQ is a fine, widely
supported alternative with strong tooling.

**GGUF (llama.cpp / Ollama).** GGUF is the answer when the deployment target is
**not a datacenter GPU**: CPU-only servers, laptops, Apple Silicon, or edge devices.
Its k-quant variants let you trade size for quality flexibly, and the llama.cpp
runtime runs almost anywhere with a small memory footprint and no Python required.
GPU throughput is lower than AWQ/GPTQ on a proper GPU, but for **local/desktop
apps, offline use, or "no GPU available"** it's unbeatable for portability — which
is exactly my situation running this test on a free Colab CPU-less-at-home setup.

**Decision in one line:** fine-tuning or quick experiments → **bitsandbytes**;
high-throughput GPU serving, quantize-once → **AWQ** (or GPTQ); CPU / edge / local /
maximum portability → **GGUF**.


## 10. Assumptions & limitations (be honest — it's scored)

- **Model:** Qwen2.5-1.5B-Instruct chosen so it runs on a free T4 with room for both
  versions; larger models (Mistral-7B) would show a bigger absolute memory gap but
  won't both fit comfortably on free-tier.
- **Throughput** is single-request, greedy decoding, batch size 1 — it measures
  per-request latency, not server throughput with batching.
- **Quality** is judged qualitatively on 5 prompts, not a benchmark like MMLU; it's
  a spot-check, not a rigorous eval.
- **VRAM** is measured with `torch.cuda.memory_allocated` (PyTorch's view), which can
  differ slightly from `nvidia-smi` because of the CUDA caching allocator.
- Numbers vary run-to-run and by which GPU Colab assigns — re-run for stable values.
